# AI Control Center – träna i molnet

Den här anteckningsboken tränar en modell på en gratis molndator med GPU och ger dig en zip-fil
som du importerar i kontrollpanelen på din egen dator (**Modeller → Importera modell**).

**Innan du börjar**

1. Slå på GPU: *Colab:* Körning → Ändra körningstyp → T4 GPU. *Kaggle:* Settings → Accelerator → GPU.
2. Kör `packa_for_molnet.bat` på din dator. Den skapar `ai-studio-moln.zip`.
3. Kör cellerna nedan uppifrån och ned.

In [ ]:
# 1. Kontrollera att en GPU finns
import torch
print("PyTorch", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "INGEN – slå på GPU enligt instruktionen ovan")

## 2. Ladda upp programmet
Colab frågar efter filen `ai-studio-moln.zip`. På Kaggle: lägg till zip-filen som dataset (Add input → Upload) innan du kör cellen.

In [ ]:
import glob, os, zipfile
if not os.path.exists("AI-studio/main.py"):
    hittade = glob.glob("/kaggle/input/**/ai-studio-moln.zip", recursive=True) + glob.glob("ai-studio-moln*.zip")
    if hittade:
        zip_fil = hittade[0]
    else:
        from google.colab import files
        zip_fil = next(iter(files.upload()))
    zipfile.ZipFile(zip_fil).extractall("AI-studio")
    # En zip nedladdad från GitHub har en extra mapp överst
    inre = [d for d in glob.glob("AI-studio/*/main.py")]
    if not os.path.exists("AI-studio/main.py") and inre:
        os.rename(os.path.dirname(inre[0]), "AI-studio-tmp"); os.rmdir("AI-studio"); os.rename("AI-studio-tmp", "AI-studio")
os.chdir("AI-studio") if os.path.basename(os.getcwd()) != "AI-studio" else None
print("Arbetsmapp:", os.getcwd())

## 3. (Valfritt) Spara allt på Google Drive
Gratis sessioner kan avbrytas. Med den här cellen sparas data och checkpoints på din Drive, så att du
kan köra om anteckningsboken och fortsätta där träningen slutade. Hoppa över cellen på Kaggle.

In [ ]:
DATAMAPP = "data"
try:
    from google.colab import drive
    drive.mount("/content/drive")
    DATAMAPP = "/content/drive/MyDrive/ai-control-center-data"
except Exception as fel:
    print("Drive används inte:", fel)
os.environ["AICC_DATA"] = DATAMAPP
print("Datamapp:", DATAMAPP)

## 4. Hämta träningsdata
Hämtar svensk text och frågor/svar från Wikipedia, Riksdagens öppna data och SweFAQ (ca 1 GB
nedladdning, 5–10 minuter). Varje källa blir ett eget dataset med källa och licens, och allt slås ihop
till datasetet **Utökad träningsdata**. Redan nedladdade filer återanvänds.

In [ ]:
!pip -q install pyarrow psutil
!python tools/hamta_data.py

## 5. Träna
Standardvalet skapar **Lennart 2** med ca 8 miljoner parametrar – tillräckligt stor för att ha nytta av
den större datamängden (ca 12 miljoner tokens). Följ val-loss i utskriften: slutar den sjunka har modellen
lärt sig det som går. Körs cellen igen fortsätter träningen
från senaste checkpoint.

Vill du i stället träna vidare på en befintlig modell: ladda upp dess exporterade zip och använd det
andra kommandot. Tänk på att en modell med 1 miljon parametrar inte har plats för mycket ny kunskap.

In [ ]:
!python moln/trana.py --ny "Lennart 2" --parametrar 8000000 --lager 6 --heads 6 --context 256 --ordforrad 8192 --steg 15000 --batch 64 --ut ../Lennart-2.aicc-modell.zip

# Träna vidare på en befintlig modell i stället:
# !python moln/trana.py --modell-zip exempel/Lennart-tranad.aicc-modell.zip --steg 20000 --ut ../Lennart.aicc-modell.zip

## 6. Ladda ner modellen
Importera sedan filen i kontrollpanelen: **Modeller → Importera modell**.

In [ ]:
try:
    from google.colab import files
    files.download("../Lennart-2.aicc-modell.zip")
except ImportError:
    import shutil
    shutil.copy("../Lennart-2.aicc-modell.zip", "/kaggle/working/")
    print("Filen finns under Output i Kaggle.")